# 正常流与溢出

学习目标：能解释块级与行内正常流，用浮动和BFC控制影响范围，并选择保留可操作性的溢出与隐藏方式。

前置知识：HTML 文档顺序、CSS 选择器、盒模型、尺寸约束、外边距折叠的基本条件。

适用范围：默认横排文档中的流式布局；不依赖框架或 JavaScript。overflow: clip 提供 hidden 回退并说明行为差异；可访问性还需结合目标浏览器与辅助技术检查。

环境准备：[环境配置与运行](README.md)。

配套脚本：位于 scripts/06-normal-flow-and-overflow/。

1. [index.html](scripts/06-normal-flow-and-overflow/index.html)、[index.css](scripts/06-normal-flow-and-overflow/index.css)：块级与行内布局、浮动、清除与 BFC。
2. [overflow.html](scripts/06-normal-flow-and-overflow/overflow.html)、[overflow.css](scripts/06-normal-flow-and-overflow/overflow.css)：溢出裁剪、滚动容器与两轴计算对照。
3. [visibility.html](scripts/06-normal-flow-and-overflow/visibility.html)、[visibility.css](scripts/06-normal-flow-and-overflow/visibility.css)：隐藏方式、焦点与盒子生成对照。

Step 1：在已激活 Python 环境的终端中，从项目根目录进入本技术目录。

```bash
cd content/Web与应用开发/css
```

Step 2：启动本章预览服务。

```bash
python -m http.server 8101 --bind 127.0.0.1
```

Step 3：打开[本章示例首页](http://127.0.0.1:8101/scripts/06-normal-flow-and-overflow/index.html)。

保存修改后刷新页面。

Step 4：在服务终端按 Ctrl+C 停止服务。

## 1 正常流中的块与行内内容

正常流（normal flow）按文档顺序参与块级和行内排版。在本章横排模式下，普通块盒依次上下排列；即使给两块都设很小的宽度，它们也不会自动并排。

行内内容沿文字方向排列，空间不足时按断行规则换行；一行占用的矩形区域称为行盒（line box）。段落里的 &lt;span&gt; 默认就是行内元素。

非替换行内盒的 width/height 不决定它的尺寸；左右 padding、border、margin 会影响水平占用，上下装饰可以画出但不按块盒方式撑开行高，上下 margin 不产生同样的布局间距。&lt;img&gt; 等替换行内元素能有宽高，不能套用普通 &lt;span&gt; 的限制。

```html
<div class="normal-demo">
  <p id="normal-a">第一块</p>
  <p id="normal-b">第二块</p>
</div>
<p class="inline-line">正文中的 <span class="plain-inline">行内</span> 内容继续沿文字方向排列。</p>
```

```css
.normal-demo > p { width: 140px; margin: 0; border: 1px solid teal; }
.plain-inline { width: 200px; height: 80px; padding: 0 6px; border: 1px solid teal; }
/* 检查：两段仍上下排列；span 不会因为 width: 200px 而变成 200px 宽的普通块。 */
```

配套文件：[index.html](scripts/06-normal-flow-and-overflow/index.html)、[index.css](scripts/06-normal-flow-and-overflow/index.css) · [浏览器预览](http://127.0.0.1:8101/scripts/06-normal-flow-and-overflow/index.html#demo-1)

## 2 display 的外部与内部角色

display 控制盒子生成和布局类型，不改变 HTML 元素原本的语义。把 &lt;span&gt; 变成块盒不会让它变成标题或按钮。

- block/inline 描述外部显示类型，决定盒子如何参与父环境。
- flow 是内部的普通块与行内布局；flow-root 明确为内容建立独立的BFC。
- inline-block 等价于 inline flow-root：外部参与行内排列，内部是可设置宽高的独立块容器，整体作为一个行内单元。
- flex/grid 指定内部的弹性或网格布局，单独写时外部通常为 block；它们在专门章节展开。

本章使用传统单关键字，便于在旧环境运行。多关键字写法需确认支持，必要时先写对应单关键字作为回退。

```html
<div class="display-line">
  文字 <span class="inline-block-sample">独立小盒</span> 后续文字
  <span class="as-block">改为块盒的 span</span>
  末尾文字
</div>
```

```css
.inline-block-sample { display: inline-block; width: 100px; height: 40px; border: 1px solid teal; }
.as-block { display: block; width: 160px; border: 1px solid teal; }
/* 检查：小盒与文字同行并占 102×42px；as-block 在前后产生块级换行。 */
```

配套文件：[index.html](scripts/06-normal-flow-and-overflow/index.html)、[index.css](scripts/06-normal-flow-and-overflow/index.css) · [浏览器预览](http://127.0.0.1:8101/scripts/06-normal-flow-and-overflow/index.html#demo-2)

## 3 float 与 clear：文字环绕和停止环绕

float 把盒子移到包含块的一侧，并让附近行盒为它留出空间；浮动盒脱离普通流内块盒的排队方式，却仍影响文字环绕。普通段落的背景可能延伸到浮动后面，改变的是行盒可用空间，不是把段落整体缩成一列。

| 完整属性名 | 中文名称／含义 | 用途 |
| --- | --- | --- |
| float | 浮动方向 | 让盒子靠一侧并影响后续文字环绕 |
| clear | 清除浮动影响 | 将当前块移到相关先前浮动的下方 |
| display | 显示与布局类型 | 控制盒生成及内外部布局方式 |

float: left/right 分别向左右浮动，none 不浮动。clear: left/right/both 分别清除前面相应侧浮动的影响；它不删除浮动，也不把浮动元素重新放回正常流。只考虑同一BFC内相关的先前浮动。

本组的 flow-root 用于把浮动包在示例内，下一节解释机制。页面整体的多列布局通常在 Flex/Grid 中表达；本例把 float 用在它擅长的文字环绕上。

```html
<div class="float-demo">
  <div class="float-thumb">浮动</div>
  <p class="story-text"><span id="wrap-start">正文</span>从浮动旁开始，继续沿余下空间排版；浮动会挤窄附近的行盒。</p>
  <p class="story-clear">这一块明确停止环绕。</p>
</div>
```

```css
.float-demo { display: flow-root; width: 360px; max-width: 100%; border: 1px solid gray; }
.float-thumb { float: left; width: 80px; height: 80px; margin-right: 12px; background-color: lightyellow; }
.story-text, .story-clear { margin: 0; }
.story-clear { clear: both; border-top: 1px solid teal; }
/* 检查文字起点位于浮动右侧；clear 块的上边框不高于浮动的底边。 */
```

配套文件：[index.html](scripts/06-normal-flow-and-overflow/index.html)、[index.css](scripts/06-normal-flow-and-overflow/index.css) · [浏览器预览](http://127.0.0.1:8101/scripts/06-normal-flow-and-overflow/index.html#demo-3)

## 4 BFC 与 flow-root 包含内部浮动

块级格式化上下文（block formatting context，BFC）是块盒布局和浮动互相作用的一片区域。普通 display: block 不一定新建BFC；display: flow-root 是表达这个意图的直接方式。

建立BFC的常见情况还有根元素、浮动盒、绝对定位盒、inline-block，以及普通块上非 visible/clip 的 overflow。Flex/Grid 容器建立的是各自的格式化上下文，不能简单都称作BFC。

这里要观察的是父盒边框，不是浮动文字有没有显示。普通块的 auto 高度不会由仅有的浮动子盒按普通流方式撑开；建立 BFC 后，自动高度会包含参与该 BFC 的浮动。两组只改变内层的 flow-root，外层则用来隔开实验。

![同样的浮动子盒位于两组父块中，普通父块边框高2px，flow-root 父块外沿高62px。](image/illustration/06-01-float-parent-height.svg)

图 1：下方例子的父盒边界。只考虑一个浮动子盒、零内边距和 1px 边框，不代表任意复杂块的完整高度算法。

在浏览器里先找到细边框，再开关内层 display: flow-root；不要因为两组浮动内容都可见，就认为父盒高度相同。

```html
<div class="float-stage">
  <div class="float-shell"><div class="floating-child">普通父块</div></div>
</div>
<div class="float-stage">
  <div class="float-shell flow-shell"><div class="floating-child">BFC父块</div></div>
</div>
```

```css
.float-stage { display: flow-root; margin-bottom: 16px; }
.float-shell { width: 240px; border: 1px solid teal; }
.floating-child { float: left; width: 80px; height: 60px; background-color: lightyellow; }
.flow-shell { display: flow-root; }
/* 比较内层边框：普通父块外沿高 2px；flow-root 父块外沿高 62px。 */
```

配套文件：[index.html](scripts/06-normal-flow-and-overflow/index.html)、[index.css](scripts/06-normal-flow-and-overflow/index.css) · [浏览器预览](http://127.0.0.1:8101/scripts/06-normal-flow-and-overflow/index.html#demo-4)

## 5 BFC 隔开外部浮动与父子外边距

独立BFC的边框盒不会与同一外部格式化上下文中的相关浮动重叠；可用宽度不足时，整个盒子可能移到浮动下方，而不是让内部文字逐行环绕。

BFC根的外边距与内部子块外边距不折叠，内部普通兄弟之间仍可折叠。建立BFC也不是清除所有外边距，更不是像素级裁剪；flow-root 本身不会隐藏越界内容。

下面先观察完整的后续盒避开浮动，再观察有 flow-root 的父框保留首个子块的顶部间距。父框用背景而不用边框，避免边框本身干扰折叠条件。

```html
<div class="external-stage">
  <div class="external-float">外部浮动</div>
  <div class="external-content">完整盒子避开浮动</div>
</div>
<div class="bfc-margin">
  <p class="bfc-first">首个子块</p>
  <p class="bfc-second">第二个子块</p>
</div>
```

```css
.external-stage { display: flow-root; width: 360px; max-width: 100%; }
.external-float { float: left; width: 80px; height: 60px; margin-right: 12px; background-color: lightyellow; }
.external-content { display: flow-root; min-height: 60px; border: 1px solid teal; }
.bfc-margin { display: flow-root; background-color: lightyellow; }
.bfc-margin > p { margin: 0; }
.bfc-margin > .bfc-first { margin-top: 24px; margin-bottom: 20px; }
.bfc-margin > .bfc-second { margin-top: 30px; }
/* 父框内部顶部保留 24px；两段之间仍折叠成 30px，而不是 50px。 */
```

配套文件：[index.html](scripts/06-normal-flow-and-overflow/index.html)、[index.css](scripts/06-normal-flow-and-overflow/index.css) · [浏览器预览](http://127.0.0.1:8101/scripts/06-normal-flow-and-overflow/index.html#demo-5)

## 6 overflow 与滚动容器

内容超出受限的尺寸时，overflow 决定显示、裁剪与滚动方式。滚动容器（scroll container）可以通过改变滚动位置显露不同内容，其可视窗口称为滚动视口（scrollport）。

| 完整属性名 | 中文名称／含义 | 用途 |
| --- | --- | --- |
| overflow | 溢出处理简写 | 设置水平与垂直方向的处理 |
| overflow-x | 水平溢出处理 | 处理左右方向溢出 |
| overflow-y | 垂直溢出处理 | 处理上下方向溢出 |

- visible：内容可画到边界外，不把此盒变成滚动容器。
- auto：有溢出时提供滚动机制；无溢出时无需显示滚动条。
- scroll：即使内容未溢出也请求滚动机制。滚动条是否常驻、是否占布局空间还受系统和浏览器界面策略影响。
- hidden：裁剪且不提供用户直接滚动界面，但仍可通过程序滚动，聚焦内部控件等也可能改变滚动位置。
- clip：裁剪且禁止滚动，不建立滚动容器；下面先写 hidden 为旧环境回退，这仅保留裁剪，不保证“禁止程序滚动”的同等行为。

本例明确给高度 48px 和内部 120px 内容以产生溢出；仅设置 overflow 不保证一定有滚动距离。可滚动区域使用 tabindex="0" 使键盘可达，并给出可读名称。CSS裁剪不等于从可访问性树移除内容，必要文字不能只靠裁掉来隐藏。

```html
<p>visible：</p>
<div class="overflow-box overflow-visible"><div class="overflow-content">第一行<br>第二行<br>第三行<br>第四行<br>第五行</div></div>
<p>auto（可用键盘滚动）：</p>
<div class="overflow-box overflow-auto" tabindex="0" role="region" aria-label="五行阅读记录"><div class="overflow-content">第一行<br>第二行<br>第三行<br>第四行<br>第五行</div></div>
<p>scroll：</p>
<div class="overflow-box overflow-scroll"><div class="overflow-content">第一行<br>第二行<br>第三行<br>第四行<br>第五行</div></div>
<p>hidden：</p>
<div class="overflow-box overflow-hidden"><div class="overflow-content">第一行<br>第二行<br>第三行<br>第四行<br>第五行</div></div>
<p>clip（旧环境回退 hidden）：</p>
<div class="overflow-box overflow-clip"><div class="overflow-content">第一行<br>第二行<br>第三行<br>第四行<br>第五行</div></div>
```

```css
.overflow-box { width: 240px; height: 48px; border: 1px solid teal; margin-bottom: 16px; }
.overflow-content { height: 120px; background-color: lightyellow; }
.overflow-visible { overflow: visible; margin-bottom: 90px; }
.overflow-auto { overflow: auto; }
.overflow-scroll { overflow: scroll; }
.overflow-hidden { overflow: hidden; }
.overflow-clip { overflow: hidden; overflow: clip; }
.overflow-auto:focus { outline: 3px solid navy; }
/* 比较可见区域和可滚动性；visible 组额外留空以免越界文字盖住下一组。
   聚焦 auto 区域后用方向键或 End，检查最后一行能否出现。 */
```

配套文件：[overflow.html](scripts/06-normal-flow-and-overflow/overflow.html)、[overflow.css](scripts/06-normal-flow-and-overflow/overflow.css) · [浏览器预览](http://127.0.0.1:8101/scripts/06-normal-flow-and-overflow/overflow.html#demo-6)

## 7 两个方向不能任意独立组合

overflow 写一个值会用于两个方向，写两个值时依次为水平与垂直。更要看计算后的组合：

如果一边既不是 visible 也不是 clip，另一边的 visible 会计算为 auto，clip 会计算为 hidden。因此 overflow-x: hidden 搭配 overflow-y: visible，不能保证垂直方向仍自由画到盒外。

普通块上 auto/scroll/hidden 会建立独立格式化上下文，clip 本身不会；若既要 clip 又要BFC，可以组合 display: flow-root。不要为了包含浮动而随手写 overflow: hidden，因为它也会裁掉阴影或内容。

clip 的裁剪边界可用 overflow-clip-margin 调整，默认以padding边为基准，负距离无效；这是单独属性且支持细节不同，本章不依赖它来保护焦点轮廓。

```html
<div class="mixed-overflow"><div class="wide-content">两方向都有超出的内容</div></div>
<div class="clip-stage"><div class="clip-shell"><div class="clip-child">浮动</div></div></div>
<div class="clip-stage"><div class="clip-shell clip-bfc"><div class="clip-child">浮动</div></div></div>
```

```css
.mixed-overflow { width: 200px; height: 48px; overflow-x: hidden; overflow-y: visible; border: 1px solid teal; }
.wide-content { width: 320px; height: 120px; background-color: lightyellow; }
.clip-stage { display: flow-root; margin: 16px 0; }
.clip-shell { width: 240px; border: 1px solid teal; overflow: hidden; overflow: clip; }
.clip-child { float: left; width: 80px; height: 60px; background-color: lightyellow; }
.clip-bfc { display: flow-root; }
/* mixed-overflow 的 overflow-y 计算为 auto。
   支持 clip 时：普通 clip-shell 外沿仅 2px；加 flow-root 后为 62px。
   回退 hidden 时两者都建立BFC，不能把回退当成与 clip 完全相同。 */
```

配套文件：[overflow.html](scripts/06-normal-flow-and-overflow/overflow.html)、[overflow.css](scripts/06-normal-flow-and-overflow/overflow.css) · [浏览器预览](http://127.0.0.1:8101/scripts/06-normal-flow-and-overflow/overflow.html#demo-7)

## 8 三种隐藏方式的布局与焦点差异

隐藏要同时决定空间是否保留、内容是否可操作以及是否应被辅助技术读取。

- display: none 不生成该元素及后代的盒，布局不保留空间，通常从可访问性树移除；后代不能通过改自己的 display 再显现。
- visibility: hidden 保留布局位置但不绘制，元素不能获得焦点；后代显式改回 visibility: visible 时可以重新显示。默认隐藏的子树也不提供普通的屏幕阅读器浏览。
- opacity: 0 只让整组绘制透明，仍保留布局、命中测试及原来的焦点资格，也不会仅因透明就从可访问性树移除。它不是通用的“禁用”方式。

下面的透明按钮在聚焦时恢复可见，避免形成看不见的操作。用Tab从开始按钮前进，对比哪些项被跳过。对于不可见但被 aria-labelledby/aria-describedby 引用的内容，名称与描述计算还有专门规则，不能把“隐藏”概括成永远不参与任何可访问性计算。

```html
<button id="hide-start" type="button">从这里开始按 Tab</button>
<button class="hide-item hide-none" type="button">display none</button>
<button class="hide-item hide-visibility" type="button">visibility hidden</button>
<button class="hide-item hide-opacity" type="button">透明按钮，聚焦后显示</button>
<button id="hide-end" type="button">结束</button>
```

```css
.hide-item { width: 220px; height: 40px; margin: 8px; }
.hide-none { display: none; }
.hide-visibility { visibility: hidden; }
.hide-opacity { opacity: 0; }
.hide-opacity:focus { opacity: 1; outline: 3px solid navy; }
/* none 无占位；hidden 与透明按钮仍占位；Tab 跳过前两种，透明按钮聚焦后显现。 */
```

配套文件：[visibility.html](scripts/06-normal-flow-and-overflow/visibility.html)、[visibility.css](scripts/06-normal-flow-and-overflow/visibility.css) · [浏览器预览](http://127.0.0.1:8101/scripts/06-normal-flow-and-overflow/visibility.html#demo-8)

## 9 contents 与 HTML hidden 要分清

display: contents 让普通容器自身不生成盒，子元素继续参与布局，DOM中的父子关系及继承仍存在。父盒没有了，其边框、padding和背景就没有可绘制的盒；这不等于 display: none。

部分浏览器与辅助技术组合对 contents 元素自身的语义暴露有过或仍有差异。涉及列表、表格、控件或重要地标时应核查可访问性树，不把它当成无条件的包装清理工具；默认保留普通容器就是合理回退。

HTML 的 hidden 是内容状态属性，不是 CSS 的 visibility 值。普通 hidden 表示内容当前不相关，通常不呈现给用户；作者CSS仍可能覆盖其显示。hidden="until-found" 还有查找揭示的独立行为，本例只用普通 hidden，不混用两种状态。

```html
<div class="contents-box">
  <p class="contents-child">父盒消失，子内容仍在</p>
</div>
<p class="ordinary-hidden" hidden>当前不呈现的内容</p>
```

```css
.contents-box { display: contents; color: navy; border: 4px solid teal; padding: 20px; }
.contents-child { border: 1px solid gray; }
.ordinary-hidden { color: maroon; }
/* 检查：父盒的 teal 边框和 padding 不绘制，子段落仍继承 navy。
   不给 hidden 元素强设 display:block；本例保留其正常隐藏语义。 */
```

配套文件：[visibility.html](scripts/06-normal-flow-and-overflow/visibility.html)、[visibility.css](scripts/06-normal-flow-and-overflow/visibility.css) · [浏览器预览](http://127.0.0.1:8101/scripts/06-normal-flow-and-overflow/visibility.html#demo-9)

## 本章小结

- 正常流按文档顺序布局；块级、行内与行内独立盒有不同的尺寸和换行方式。
- float 改变行盒环绕，clear 让后续块避开相关浮动；flow-root 明确建立BFC。
- BFC包含内部浮动并隔开父子外边距，但不会自动裁剪，也不取消内部兄弟折叠。
- hidden 可程序滚动，clip 禁止滚动；两个方向的overflow组合有计算值转换。
- 隐藏要兼顾布局、焦点与可访问性，透明、无盒与不可见不是同一件事。

## 练习

在 scripts/06-normal-flow-and-overflow/ 的配套文件中操作，先记录原值，练习后恢复。

（1）把 .plain-inline 改成 display: inline-block，检查 width/height 开始影响独立盒尺寸，观察行高与换行变化。

（2）取消 .story-clear 的 clear: both，检查它是否回到仍受浮动影响的位置；再恢复该声明。

（3）在 .bfc-margin 内把第二段 margin-top 改为 10px，检查兄弟间距变为 20px，父框顶部仍保留24px。

（4）将 .mixed-overflow 的 overflow-x 改为 visible，保留 overflow-y: visible，检查其不再产生自身滚动；恢复后比较计算值。

（5）用Tab检查 visibility.html；确认不会在不可见状态中停留。再临时把普通 hidden 元素设为 display: block，观察作者样式对其显示的影响并立即恢复。

### 提示

先确认实际 DOM 和尺寸，再看参与的格式化上下文；隐藏题不能仅凭截图判断是否仍可聚焦或被辅助技术读取。

## 参考与引用来源

- W3C：[CSS 2.2 §10.6.7](https://www.w3.org/TR/CSS22/visudet.html#root-height) 的 BFC 根自动高度与浮动后代；[CSS 2.2 §9.4–9.5](https://www.w3.org/TR/CSS22/visuren.html#normal-flow) 的正常流、行盒、浮动与clear；[CSS Display Level 3 §2](https://www.w3.org/TR/css-display-3/#the-display-properties) 的内外部类型、flow-root与盒生成；[CSS Overflow Level 3 §3.1–3.2](https://www.w3.org/TR/css-overflow-3/#overflow-properties) 的滚动容器、hidden/clip、方向转换和裁剪边界。
- MDN：[Block and inline layout](https://developer.mozilla.org/en-US/docs/Web/CSS/Guides/Display/Block_and_inline_layout) 的正常流与尺寸；[display](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/display#grouped_values) 的单关键字与双关键字、none与contents；[Block formatting context](https://developer.mozilla.org/en-US/docs/Web/CSS/Guides/Display/Block_formatting_context#examples) 的建立条件、包含内部与排除外部浮动、隔开父子折叠；[float](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/float#examples)、[clear](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/clear#description) 的浮动和清除范围；[overflow](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/overflow#description) 的溢出条件和交互；[overflow-clip-margin](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/overflow-clip-margin) 的裁剪边界与支持；[visibility](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/visibility#values)、[opacity](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/opacity#description) 与 [display 的 Accessibility](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/display#accessibility) 的隐藏、焦点与语义边界。
- WHATWG HTML：[The hidden attribute](https://html.spec.whatwg.org/multipage/interaction.html#the-hidden-attribute) 的普通隐藏、until-found及作者CSS覆盖。